# 09 - Análise Trimestral: Nov 2025 → Fev 2026

Análise setorizada do último trimestre de dados para identificar:
- Mudanças no mapa de calor (hora x dia) mês a mês
- Drift na % de LOWs e média do multiplicador
- Padrões semanais e diários de volatilidade
- Distribuição de streaks e probabilidades condicionais por período
- Indicadores para estratégia adaptativa

In [ ]:
import sys
sys.path.insert(0, '.')
from config_analysis import *

import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = 'plotly_dark'

# Cores
CYAN = '#00f0ff'
MAGENTA = '#ff00ff'
GREEN = '#00ff88'
ORANGE = '#ff8800'
RED = '#ff3366'
YELLOW = '#ffff00'
PURPLE = '#aa66ff'
COLORS_MONTHS = [CYAN, MAGENTA, GREEN, ORANGE]

# Período de análise
DATE_START = '2025-11-01'
DATE_END = '2026-02-28'

print(f'Período: {DATE_START} a {DATE_END}')

In [ ]:
# Carregar dados do período
conn = duckdb.connect()
df = conn.execute(f"""
    SELECT
        id,
        date,
        time,
        numericResult as multiplicador,
        result,
        type as tipo
    FROM sqlite_scan('{DB_PATH}', 'crash_rounds')
    WHERE date >= '{DATE_START}'
    ORDER BY date ASC
""").fetchdf()
conn.close()

# Parse datas
df['date'] = pd.to_datetime(df['date'])
df['dia'] = df['date'].dt.date
df['hora'] = df['date'].dt.hour
df['dia_semana'] = df['date'].dt.dayofweek
df['dia_semana_nome'] = df['date'].dt.day_name()
df['mes'] = df['date'].dt.strftime('%Y-%m')
df['semana'] = df['date'].dt.strftime('%Y-W%W')
df['quinzena'] = df['date'].apply(
    lambda x: f"{x.strftime('%Y-%m')}-Q1" if x.day <= 15 else f"{x.strftime('%Y-%m')}-Q2"
)

# Features básicas
df['is_low'] = (df['multiplicador'] < LOW_THRESHOLD).astype(int)
df['is_high'] = 1 - df['is_low']

# Streaks de LOWs consecutivos
df['low_group'] = (df['is_low'] != df['is_low'].shift()).cumsum()
df['low_streak'] = df.groupby('low_group')['is_low'].cumsum() * df['is_low']

# Rolling stats (janelas curtas para análise tática)
for w in [50, 100, 250, 500]:
    df[f'rolling_mean_{w}'] = df['multiplicador'].rolling(w, min_periods=w).mean()
    df[f'rolling_pct_low_{w}'] = df['is_low'].rolling(w, min_periods=w).mean() * 100
    df[f'rolling_std_{w}'] = df['multiplicador'].rolling(w, min_periods=w).std()

meses = sorted(df['mes'].unique())
print(f'Registros: {len(df):,}')
print(f'Período: {df["date"].min().strftime("%Y-%m-%d %H:%M")} → {df["date"].max().strftime("%Y-%m-%d %H:%M")}')
print(f'Meses: {meses}')
print(f'\nResumo por mês:')
for m in meses:
    sub = df[df['mes'] == m]
    print(f'  {m}: {len(sub):>7,} rounds | '
          f'média={sub["multiplicador"].mean():.4f}x | '
          f'mediana={sub["multiplicador"].median():.2f}x | '
          f'%LOW={sub["is_low"].mean()*100:.2f}% | '
          f'max_streak={sub["low_streak"].max()}')

---
## 1. Heatmap: % LOW por Hora x Dia da Semana — Comparação Mensal

Mesmo gráfico do notebook 02, mas **um por mês** para ver se o padrão hora-dia muda.

In [ ]:
dias_nomes = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'% LOW — {m}' for m in meses],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

# Range global para consistência de cor
all_vals = []
heatmaps = {}
for m in meses:
    sub = df[df['mes'] == m]
    h = sub.groupby(['dia_semana', 'hora'])['is_low'].mean().unstack(fill_value=np.nan) * 100
    heatmaps[m] = h
    all_vals.extend(h.values.flatten()[~np.isnan(h.values.flatten())])

vmin = np.percentile(all_vals, 2)
vmax = np.percentile(all_vals, 98)

for idx, m in enumerate(meses):
    row = idx // 2 + 1
    col = idx % 2 + 1
    h = heatmaps[m]
    
    fig.add_trace(go.Heatmap(
        z=h.values,
        x=list(range(24)),
        y=dias_nomes,
        colorscale='RdYlGn_r',
        zmin=vmin, zmax=vmax,
        text=np.round(h.values, 1),
        texttemplate='%{text}',
        textfont={'size': 8},
        showscale=(idx == 0),
        colorbar=dict(title='% LOW') if idx == 0 else None,
    ), row=row, col=col)

fig.update_layout(
    height=650, width=1100,
    title_text='Heatmap % LOW (Hora x Dia) — Evolução Mensal',
)
fig.show()

In [ ]:
# Heatmap de DIFERENÇA: cada mês vs média do trimestre
global_heat = df.groupby(['dia_semana', 'hora'])['is_low'].mean().unstack(fill_value=np.nan) * 100

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'Desvio da média — {m}' for m in meses],
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

for idx, m in enumerate(meses):
    row = idx // 2 + 1
    col = idx % 2 + 1
    diff = heatmaps[m] - global_heat
    
    fig.add_trace(go.Heatmap(
        z=diff.values,
        x=list(range(24)),
        y=dias_nomes,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-5, zmax=5,
        text=np.round(diff.values, 1),
        texttemplate='%{text}',
        textfont={'size': 8},
        showscale=(idx == 0),
        colorbar=dict(title='Desvio pp') if idx == 0 else None,
    ), row=row, col=col)

fig.update_layout(
    height=650, width=1100,
    title_text='Desvio do Heatmap vs Média Trimestral (pp)',
)
fig.show()

# Quantificar: quais slots mudaram mais?
print('\nSlots com maior variação entre meses (std > 2pp):')
for dia in range(7):
    for hora in range(24):
        vals = [heatmaps[m].loc[dia, hora] if dia in heatmaps[m].index and hora in heatmaps[m].columns else np.nan for m in meses]
        vals = [v for v in vals if not np.isnan(v)]
        if len(vals) >= 3 and np.std(vals) > 2.0:
            print(f'  {dias_nomes[dia]} {hora:02d}h: ' + ' | '.join(f'{v:.1f}%' for v in vals) + f'  (std={np.std(vals):.1f})')

---
## 2. Evolução Semanal — Tendência e Drift

In [ ]:
# Métricas por semana
weekly = df.groupby('semana').agg(
    total=('multiplicador', 'count'),
    media=('multiplicador', 'mean'),
    mediana=('multiplicador', 'median'),
    std=('multiplicador', 'std'),
    pct_low=('is_low', 'mean'),
    max_streak=('low_streak', 'max'),
    pct_above_10x=('multiplicador', lambda x: (x >= 10).mean()),
).reset_index()
weekly['pct_low'] *= 100
weekly['pct_above_10x'] *= 100

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=(
        'Média Semanal do Multiplicador',
        '% LOWs Semanal',
        'Max Streak LOW por Semana',
        'Volatilidade (std) Semanal',
    ),
    shared_xaxes=True,
    vertical_spacing=0.06,
)

fig.add_trace(go.Scatter(
    x=weekly['semana'], y=weekly['media'],
    mode='lines+markers', name='Média',
    line=dict(color=CYAN, width=2),
    marker=dict(size=6),
), row=1, col=1)
fig.add_hline(y=df['multiplicador'].mean(), line_dash='dash',
              line_color=YELLOW, row=1, col=1,
              annotation_text=f'Média trimestre: {df["multiplicador"].mean():.2f}x')

fig.add_trace(go.Bar(
    x=weekly['semana'], y=weekly['pct_low'],
    name='% LOW', marker_color=MAGENTA,
), row=2, col=1)
fig.add_hline(y=df['is_low'].mean()*100, line_dash='dash',
              line_color=YELLOW, row=2, col=1)

fig.add_trace(go.Bar(
    x=weekly['semana'], y=weekly['max_streak'],
    name='Max Streak', marker_color=RED,
), row=3, col=1)
fig.add_hline(y=TRAGEDY_STREAK, line_dash='dash',
              line_color=YELLOW, row=3, col=1,
              annotation_text='Tragédia (12+)')

fig.add_trace(go.Scatter(
    x=weekly['semana'], y=weekly['std'],
    mode='lines+markers', name='Std',
    line=dict(color=ORANGE, width=2),
), row=4, col=1)

fig.update_layout(height=1000, title_text='Evolução Semanal — Trimestre Nov/25-Fev/26')
fig.show()

print(weekly.to_string(index=False))

---
## 3. Evolução Diária — Detecção de Anomalias

In [ ]:
# Métricas por dia
daily = df.groupby('dia').agg(
    total=('multiplicador', 'count'),
    media=('multiplicador', 'mean'),
    mediana=('multiplicador', 'median'),
    std=('multiplicador', 'std'),
    pct_low=('is_low', 'mean'),
    max_streak=('low_streak', 'max'),
).reset_index()
daily['pct_low'] *= 100
daily['dia'] = pd.to_datetime(daily['dia'])

# Bandas de Bollinger da % LOW (média móvel 7 dias ± 2σ)
daily['pct_low_ma7'] = daily['pct_low'].rolling(7, min_periods=3).mean()
daily['pct_low_std7'] = daily['pct_low'].rolling(7, min_periods=3).std()
daily['upper_band'] = daily['pct_low_ma7'] + 2 * daily['pct_low_std7']
daily['lower_band'] = daily['pct_low_ma7'] - 2 * daily['pct_low_std7']

# Anomalias: dias fora das bandas
daily['anomalia'] = (
    (daily['pct_low'] > daily['upper_band']) |
    (daily['pct_low'] < daily['lower_band'])
)

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        '% LOWs Diário com Bandas de Bollinger (7d)',
        'Média Diária do Multiplicador',
        'Max Streak LOW por Dia',
    ),
    shared_xaxes=True,
    vertical_spacing=0.06,
)

# % LOW com bandas
fig.add_trace(go.Scatter(
    x=daily['dia'], y=daily['pct_low'],
    mode='markers', name='% LOW diário',
    marker=dict(
        color=[RED if a else CYAN for a in daily['anomalia']],
        size=[8 if a else 4 for a in daily['anomalia']],
    ),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily['dia'], y=daily['pct_low_ma7'],
    mode='lines', name='MA 7d',
    line=dict(color=YELLOW, width=2),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily['dia'], y=daily['upper_band'],
    mode='lines', name='Upper Band',
    line=dict(color=ORANGE, width=1, dash='dash'),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily['dia'], y=daily['lower_band'],
    mode='lines', name='Lower Band',
    line=dict(color=ORANGE, width=1, dash='dash'),
    fill='tonexty', fillcolor='rgba(255,136,0,0.1)',
), row=1, col=1)

# Média diária
fig.add_trace(go.Scatter(
    x=daily['dia'], y=daily['media'],
    mode='lines+markers', name='Média mult.',
    line=dict(color=GREEN, width=1.5),
    marker=dict(size=3),
), row=2, col=1)

# Max streak
streak_colors = [RED if s >= TRAGEDY_STREAK else CYAN for s in daily['max_streak']]
fig.add_trace(go.Bar(
    x=daily['dia'], y=daily['max_streak'],
    name='Max Streak', marker_color=streak_colors,
), row=3, col=1)
fig.add_hline(y=TRAGEDY_STREAK, line_dash='dash',
              line_color=YELLOW, row=3, col=1)

fig.update_layout(height=900, title_text='Evolução Diária — Trimestre')
fig.show()

anomalias = daily[daily['anomalia']]
print(f'\nDias anômalos ({len(anomalias)}):')
if len(anomalias) > 0:
    for _, row in anomalias.iterrows():
        direcao = '↑ ALTO' if row['pct_low'] > row['pct_low_ma7'] else '↓ BAIXO'
        print(f'  {row["dia"].strftime("%Y-%m-%d")}: {row["pct_low"]:.1f}% ({direcao}) | '
              f'MA7={row["pct_low_ma7"]:.1f}% | streak_max={row["max_streak"]:.0f}')

---
## 4. Distribuição de Streaks — Comparação Mensal

In [ ]:
# Extrair sequências completas de LOWs por mês
streak_data = []

for m in meses:
    sub = df[df['mes'] == m].copy()
    # Fim de cada sequência LOW
    ends = sub[(sub['low_streak'] > 0) & (sub['is_low'].shift(-1, fill_value=0) == 0)]
    lengths = ends['low_streak'].values
    for l in lengths:
        streak_data.append({'mes': m, 'streak_len': int(l)})

streak_df = pd.DataFrame(streak_data)

# Distribuição comparativa
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribuição de Streaks (1-15)', 'Streaks 6+ (zona de gatilho)'),
)

for i, m in enumerate(meses):
    sub = streak_df[streak_df['mes'] == m]
    counts = sub['streak_len'].value_counts().sort_index()
    
    fig.add_trace(go.Bar(
        x=counts.index[:15], y=counts.values[:15],
        name=m, marker_color=COLORS_MONTHS[i],
        opacity=0.7,
    ), row=1, col=1)

# Foco na zona de gatilho (6+)
trigger_data = []
for m in meses:
    sub = streak_df[(streak_df['mes'] == m) & (streak_df['streak_len'] >= 6)]
    total_seqs = len(streak_df[streak_df['mes'] == m])
    trigger_data.append({
        'mes': m,
        'seqs_6plus': len(sub),
        'pct_6plus': len(sub) / total_seqs * 100 if total_seqs > 0 else 0,
        'avg_len_6plus': sub['streak_len'].mean() if len(sub) > 0 else 0,
        'max_len': sub['streak_len'].max() if len(sub) > 0 else 0,
    })

trigger_df = pd.DataFrame(trigger_data)

fig.add_trace(go.Bar(
    x=trigger_df['mes'], y=trigger_df['seqs_6plus'],
    name='Qtd 6+', marker_color=RED,
    text=[f'{p:.1f}%' for p in trigger_df['pct_6plus']],
    textposition='outside',
), row=1, col=2)

fig.update_layout(height=500, title_text='Distribuição de Streaks LOW — Por Mês', barmode='group')
fig.show()

print('\nEstatísticas de streaks 6+ por mês:')
print(trigger_df.to_string(index=False))

---
## 5. Probabilidade Condicional — Comparação Mensal

P(próximo=LOW | streak atual = k) — existe drift entre meses?

In [ ]:
# Probabilidade condicional por mês
max_pos = 15
next_is_low = df['is_low'].shift(-1)

fig = go.Figure()

for i, m in enumerate(meses):
    mask_mes = df['mes'] == m
    probs = []
    positions = []
    for pos in range(0, max_pos + 1):
        mask = mask_mes & (df['low_streak'] == pos)
        n = mask.sum()
        if n < 50:  # amostra mínima
            continue
        p = next_is_low[mask].mean() * 100
        probs.append(p)
        positions.append(pos)
    
    fig.add_trace(go.Scatter(
        x=positions, y=probs,
        mode='lines+markers', name=m,
        line=dict(color=COLORS_MONTHS[i], width=2),
        marker=dict(size=7),
    ))

# Baseline
baseline = df['is_low'].mean() * 100
fig.add_hline(y=baseline, line_dash='dash', line_color=YELLOW,
              annotation_text=f'Baseline: {baseline:.1f}%')

fig.update_layout(
    height=550,
    title_text='P(próximo=LOW) por Posição na Streak — Comparação Mensal',
    xaxis_title='Posição na streak LOW',
    yaxis_title='Probabilidade (%)',
)
fig.show()

# Tabela numérica
print('\nP(próximo=LOW) por posição e mês:')
header = f'{"Pos":>4} ' + ' '.join(f'{m:>10}' for m in meses)
print(header)
print('-' * len(header))
for pos in range(0, max_pos + 1):
    vals = []
    for m in meses:
        mask = (df['mes'] == m) & (df['low_streak'] == pos)
        n = mask.sum()
        if n < 50:
            vals.append('      n/a')
        else:
            p = next_is_low[mask].mean() * 100
            vals.append(f'{p:>9.1f}%')
    print(f'{pos:>4} ' + ' '.join(vals))

---
## 6. Heatmap por Hora — Evolução Quinzenal

Visão mais granular: como a % LOW por hora mudou a cada quinzena.

In [ ]:
quinzenas = sorted(df['quinzena'].unique())
n_q = len(quinzenas)

# Calcular % LOW por hora para cada quinzena
q_heatmaps = {}
for q in quinzenas:
    sub = df[df['quinzena'] == q]
    h = sub.groupby('hora')['is_low'].mean() * 100
    q_heatmaps[q] = h

# Montar matriz: quinzenas x horas
matrix = pd.DataFrame(q_heatmaps).T
matrix = matrix.reindex(columns=range(24))

fig = go.Figure(go.Heatmap(
    z=matrix.values,
    x=[f'{h}h' for h in range(24)],
    y=matrix.index.tolist(),
    colorscale='RdYlGn_r',
    text=np.round(matrix.values, 1),
    texttemplate='%{text}',
    textfont={'size': 9},
    colorbar=dict(title='% LOW'),
))

fig.update_layout(
    height=400, width=1100,
    title_text='% LOW por Hora — Evolução Quinzenal',
    yaxis=dict(autorange='reversed'),
)
fig.show()

# Variação por hora entre quinzenas
hora_std = matrix.std(axis=0)
print('\nVariação (std) da % LOW por hora entre quinzenas:')
for hora in range(24):
    bar = '█' * int(hora_std.iloc[hora] * 2)
    print(f'  {hora:02d}h: {hora_std.iloc[hora]:.2f}pp  {bar}')

---
## 7. Volatilidade e Regime — Rolling Metrics

In [ ]:
# Rolling % LOW com janelas de 500 e 250 rounds
# (mostra mudanças de regime mais claramente que janela diária)

# Amostrar para performance (plotar cada 50 pontos)
sample = df.iloc[500::50].copy()  # após warmup da janela

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        'Rolling % LOW (janela 500 rounds)',
        'Rolling Média do Multiplicador (janela 500)',
        'Rolling Volatilidade/Std (janela 500)',
    ),
    shared_xaxes=True,
    vertical_spacing=0.06,
)

fig.add_trace(go.Scatter(
    x=sample['date'], y=sample['rolling_pct_low_500'],
    mode='lines', name='% LOW (500r)',
    line=dict(color=MAGENTA, width=1.5),
), row=1, col=1)
fig.add_hline(y=df['is_low'].mean()*100, line_dash='dash',
              line_color=YELLOW, row=1, col=1)

fig.add_trace(go.Scatter(
    x=sample['date'], y=sample['rolling_mean_500'],
    mode='lines', name='Média (500r)',
    line=dict(color=CYAN, width=1.5),
), row=2, col=1)
fig.add_hline(y=df['multiplicador'].mean(), line_dash='dash',
              line_color=YELLOW, row=2, col=1)

fig.add_trace(go.Scatter(
    x=sample['date'], y=sample['rolling_std_500'],
    mode='lines', name='Std (500r)',
    line=dict(color=ORANGE, width=1.5),
), row=3, col=1)

fig.update_layout(height=800, title_text='Regime Detection — Rolling Metrics (500 rounds)')
fig.show()

---
## 8. Análise por Faixa Horária (Turnos)

Comparar turnos: Madrugada (0-5), Manhã (6-11), Tarde (12-17), Noite (18-23)

In [ ]:
def get_turno(hora):
    if hora < 6: return 'Madrugada (0-5h)'
    elif hora < 12: return 'Manhã (6-11h)'
    elif hora < 18: return 'Tarde (12-17h)'
    else: return 'Noite (18-23h)'

df['turno'] = df['hora'].apply(get_turno)

# Análise por turno e mês
turno_mes = df.groupby(['mes', 'turno']).agg(
    total=('multiplicador', 'count'),
    media=('multiplicador', 'mean'),
    pct_low=('is_low', 'mean'),
    max_streak=('low_streak', 'max'),
).reset_index()
turno_mes['pct_low'] *= 100

turnos_order = ['Madrugada (0-5h)', 'Manhã (6-11h)', 'Tarde (12-17h)', 'Noite (18-23h)']
turno_colors = {'Madrugada (0-5h)': PURPLE, 'Manhã (6-11h)': CYAN, 'Tarde (12-17h)': GREEN, 'Noite (18-23h)': ORANGE}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('% LOW por Turno e Mês', 'Média Multiplicador por Turno e Mês'),
)

for turno in turnos_order:
    sub = turno_mes[turno_mes['turno'] == turno]
    fig.add_trace(go.Scatter(
        x=sub['mes'], y=sub['pct_low'],
        mode='lines+markers', name=turno,
        line=dict(color=turno_colors[turno], width=2),
        marker=dict(size=8),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sub['mes'], y=sub['media'],
        mode='lines+markers', name=turno,
        line=dict(color=turno_colors[turno], width=2),
        marker=dict(size=8),
        showlegend=False,
    ), row=1, col=2)

fig.update_layout(height=450, title_text='Desempenho por Turno — Evolução Mensal')
fig.show()

# Tabela resumo
print('\n% LOW por Turno e Mês:')
pivot = turno_mes.pivot(index='turno', columns='mes', values='pct_low')
pivot = pivot.reindex(turnos_order)
print(pivot.round(2).to_string())

print('\nMédia Multiplicador por Turno e Mês:')
pivot2 = turno_mes.pivot(index='turno', columns='mes', values='media')
pivot2 = pivot2.reindex(turnos_order)
print(pivot2.round(4).to_string())

---
## 9. Diagnóstico: Mudou Algo?

Testes estatísticos para detectar drift entre meses.

In [ ]:
from scipy import stats as sp_stats

# Teste qui-quadrado: proporção de LOWs mudou entre meses?
print('=' * 60)
print('TESTES DE DRIFT ENTRE MESES')
print('=' * 60)

# 1. Proporção de LOWs (chi-squared)
contingency = pd.crosstab(df['mes'], df['is_low'])
chi2, p_chi2, dof, expected = sp_stats.chi2_contingency(contingency)
print(f'\n1. Chi² para proporção LOW entre meses:')
print(f'   chi² = {chi2:.4f}, p = {p_chi2:.6f}, dof = {dof}')
print(f'   Resultado: {"SIGNIFICATIVO" if p_chi2 < 0.05 else "NÃO significativo"} (α=0.05)')

# 2. Kruskal-Wallis para distribuição do multiplicador
groups = [df[df['mes'] == m]['multiplicador'].values for m in meses if len(df[df['mes'] == m]) > 100]
if len(groups) >= 2:
    kw_stat, p_kw = sp_stats.kruskal(*groups)
    print(f'\n2. Kruskal-Wallis para distribuição multiplicador entre meses:')
    print(f'   H = {kw_stat:.4f}, p = {p_kw:.6f}')
    print(f'   Resultado: {"SIGNIFICATIVO" if p_kw < 0.05 else "NÃO significativo"} (α=0.05)')

# 3. Mann-Whitney pairwise entre meses consecutivos
print(f'\n3. Mann-Whitney U (pairwise entre meses consecutivos):')
for i in range(len(meses) - 1):
    m1, m2 = meses[i], meses[i+1]
    g1 = df[df['mes'] == m1]['multiplicador'].values
    g2 = df[df['mes'] == m2]['multiplicador'].values
    u_stat, p_mw = sp_stats.mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '*' if p_mw < 0.05 else ' '
    print(f'   {m1} vs {m2}: U={u_stat:,.0f}, p={p_mw:.6f} {sig}')

# 4. Proporção de LOWs pairwise (z-test)
print(f'\n4. Z-test para proporção LOWs (pairwise):')
for i in range(len(meses) - 1):
    m1, m2 = meses[i], meses[i+1]
    n1 = len(df[df['mes'] == m1])
    n2 = len(df[df['mes'] == m2])
    p1 = df[df['mes'] == m1]['is_low'].mean()
    p2 = df[df['mes'] == m2]['is_low'].mean()
    p_pool = (p1 * n1 + p2 * n2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p1 - p2) / se if se > 0 else 0
    p_z = 2 * (1 - sp_stats.norm.cdf(abs(z)))
    sig = '*' if p_z < 0.05 else ' '
    print(f'   {m1} ({p1*100:.2f}%) vs {m2} ({p2*100:.2f}%): z={z:.3f}, p={p_z:.4f} {sig}')

# 5. Estabilidade da probabilidade condicional P(próximo=LOW | streak=6)
# Usar next_is_low do dataframe ORIGINAL (não do subset filtrado)
next_low = df['is_low'].shift(-1)
print(f'\n5. Estabilidade da P(próximo=LOW | streak=6) entre meses:')
for m in meses:
    mask = (df['mes'] == m) & (df['low_streak'] == 6)
    n = mask.sum()
    if n >= 30:
        p = next_low[mask].dropna().mean() * 100
        ci = 1.96 * np.sqrt(p/100 * (1-p/100) / n) * 100
        print(f'   {m}: P(next=LOW)={p:.1f}% ± {ci:.1f}pp (n={n})')
    else:
        print(f'   {m}: n={n} (amostra insuficiente)')

---
## 10. Resumo Executivo e Recomendação de Estratégia

In [ ]:
print('=' * 70)
print('RESUMO EXECUTIVO — TRIMESTRE NOV/2025 A FEV/2026')
print('=' * 70)

print(f"""
DADOS:
  Período: {df['date'].min().strftime('%Y-%m-%d')} a {df['date'].max().strftime('%Y-%m-%d')}
  Total:   {len(df):,} rounds

MÉTRICAS GLOBAIS DO TRIMESTRE:
  % LOW:    {df['is_low'].mean()*100:.2f}%
  Média:    {df['multiplicador'].mean():.4f}x
  Mediana:  {df['multiplicador'].median():.2f}x
  Std:      {df['multiplicador'].std():.4f}
  Max:      {df['multiplicador'].max():.1f}x
  Max Streak LOW: {df['low_streak'].max()}
""")

print('COMPARAÇÃO MENSAL:')
for m in meses:
    sub = df[df['mes'] == m]
    ends = sub[(sub['low_streak'] > 0) & (sub['is_low'].shift(-1, fill_value=0) == 0)]
    seqs_6plus = (ends['low_streak'] >= 6).sum()
    total_seqs = len(ends)
    print(f'  {m}:')
    print(f'    Rounds: {len(sub):>7,} | %LOW: {sub["is_low"].mean()*100:.2f}% | '
          f'Média: {sub["multiplicador"].mean():.4f}x | '
          f'MaxStreak: {sub["low_streak"].max()} | '
          f'Triggers(6+): {seqs_6plus}/{total_seqs}')

print(f"""
DIAGNÓSTICO:
  O trimestre será analisado quanto a:
  - Drift na proporção de LOWs (chi², z-test)
  - Mudanças no heatmap hora x dia
  - Estabilidade da probabilidade condicional P(LOW|streak=k)
  - Regimes de volatilidade

INDICADORES PARA ESTRATÉGIA:
  → Ver resultados dos testes estatísticos na seção 9
  → Ver heatmaps de diferença na seção 1
  → Ver evolução da P(LOW|streak) na seção 5
""")